# Feature Emergence Tracking

Track when specific features emerge across layers: probe trajectories,
token predictions, component contributions, interference, and summary.

## Why This Matters

Feature emergence tracking investigates the interpretable directions that models use internally. Understanding features — the natural units of neural computation — is essential for moving beyond neuron-level analysis to a true understanding of what models represent.

**Key references:**
- [Bricken et al. (2023) "Towards Monosemanticity"](https://transformer-circuits.pub/2023/monosemantic-features/index.html) — Sparse autoencoders find interpretable features
- [Cunningham et al. (2023) "Sparse Autoencoders Find Highly Interpretable Features"](https://arxiv.org/abs/2309.08600) — SAE features in larger language models

In [ ]:
import jax
import jax.numpy as jnp
from irtk import HookedTransformer, HookedTransformerConfig
from irtk.feature_emergence_tracking import (
    feature_probe_trajectory, token_feature_emergence,
    component_feature_contribution, feature_interference,
    feature_emergence_summary,
)

cfg = HookedTransformerConfig(
    n_layers=4, d_model=32, n_ctx=64, d_head=8, n_heads=4, d_vocab=100,
)
model = HookedTransformer(cfg)
key = jax.random.PRNGKey(42)
leaves, treedef = jax.tree.flatten(model)
new_leaves = []
for leaf in leaves:
    if isinstance(leaf, jnp.ndarray) and leaf.dtype == jnp.float32:
        key, subkey = jax.random.split(key)
        new_leaves.append(jax.random.normal(subkey, leaf.shape) * 0.1)
    else:
        new_leaves.append(leaf)
model = jax.tree.unflatten(treedef, new_leaves)
tokens = jnp.array([1, 15, 30, 45, 60, 75, 90])
print('Model ready')

## Feature Probe Trajectory

Track how strongly a direction is represented at each layer.

In [ ]:
direction = jax.random.normal(jax.random.PRNGKey(0), (32,))
result = feature_probe_trajectory(model, tokens, direction)
print(f"Emergence layer: {result['emergence_layer']}")
print(f"Max projection: {result['max_projection']:.4f}\n")
for p in result['per_layer']:
    bar = '█' * int(p['abs_projection'] * 20)
    print(f"  Layer {p['layer']}: proj={p['projection']:+.4f}, "
          f"frac={p['fraction_of_norm']:.4f} {bar}")

## Token Feature Emergence

Which tokens become predicted across layers?

In [ ]:
result = token_feature_emergence(model, tokens)
print(f"Final prediction: token {result['final_prediction']}")
print(f"Commit layer: {result['commit_layer']}\n")
for p in result['per_layer']:
    top3 = ', '.join(f'tok{t["token_id"]}({t["probability"]:.3f})' for t in p['top_tokens'][:3])
    print(f"  Layer {p['layer']}: H={p['entropy']:.3f}, top={p['top_token']} | {top3}")

## Component Feature Contribution

Does attention or MLP contribute more to a feature direction?

In [ ]:
direction = jax.random.normal(jax.random.PRNGKey(0), (32,))
result = component_feature_contribution(model, tokens, direction)
print(f"Total attn: {result['total_attn']:+.4f}")
print(f"Total mlp:  {result['total_mlp']:+.4f}\n")
for p in result['per_layer']:
    print(f"  Layer {p['layer']}: attn={p['attn_contribution']:+.4f}, "
          f"mlp={p['mlp_contribution']:+.4f} [{p['dominant']}]")

## Feature Interference

Do multiple features compete for representation capacity?

In [ ]:
d1 = jax.random.normal(jax.random.PRNGKey(0), (32,))
d2 = jax.random.normal(jax.random.PRNGKey(1), (32,))
d3 = jax.random.normal(jax.random.PRNGKey(2), (32,))
result = feature_interference(model, tokens, [d1, d2, d3])
print(f"Directions: {result['n_directions']}")
print(f"Mean interference: {result['mean_interference']:.4f}")
print(f"High interference: {result['high_interference']}\n")
for p in result['per_layer']:
    projs = ', '.join(f'{v:+.3f}' for v in p['projections'])
    print(f"  Layer {p['layer']}: projections=[{projs}]")

## Emergence Summary

Cross-layer overview of feature emergence.

In [ ]:
result = feature_emergence_summary(model, tokens)
print(f"Entropy reduction: {result['entropy_reduction']:.4f}\n")
for p in result['per_layer']:
    bar = '█' * int(p['max_prob'] * 40)
    print(f"  Layer {p['layer']}: top={p['top_token']:3d}, "
          f"prob={p['max_prob']:.4f}, H={p['entropy']:.3f}, "
          f"new_info={p['new_info_fraction']:.4f} {bar}")